# STEP 1: Loading, Inspecting, and Segmenting Texts

In [37]:
from pathlib import Path

TEXT_DIR = Path("texts")

# Collect all .txt files
files = sorted(TEXT_DIR.glob("*.txt"))

print(f"Found {len(files)} .txt files.")

# Print the first few filenames
print("First 10 files:")
for f in files[:10]:
    print(" ", f)

from pathlib import Path

TEXT_DIR = Path("texts")

# Collect all .txt files
files = sorted(TEXT_DIR.glob("*.txt"))

print(f"Found {len(files)} .txt files.")

# Print the first few filenames
print("First 10 files:")
for f in files[:10]:
    print(" ", f)

# Read one example file
example_file = files[0]

with open(example_file, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

print("\nReading file:")
print(example_file)
print("-" * 40)
print("Number of characters:", len(text))
print("\nFirst 1,000 characters:\n")
print(text[:1000])

Found 571 .txt files.
First 10 files:
  texts\A00169.txt
  texts\A00183.txt
  texts\A00188.txt
  texts\A00206.txt
  texts\A00250.txt
  texts\A00263.txt
  texts\A00419.txt
  texts\A00430.txt
  texts\A00448.txt
  texts\A00503.txt
Found 571 .txt files.
First 10 files:
  texts\A00169.txt
  texts\A00183.txt
  texts\A00188.txt
  texts\A00206.txt
  texts\A00250.txt
  texts\A00263.txt
  texts\A00419.txt
  texts\A00430.txt
  texts\A00448.txt
  texts\A00503.txt

Reading file:
texts\A00169.txt
----------------------------------------
Number of characters: 24025

First 1,000 characters:

article to be inquire of in the diocese of chester in the ordinary visitation hold 1605. titul i. article concern religion prayer and sacrament first whether there be any abide in heretical opinion or resort to your parish that be know to have defend or maintain since the twenty day of march 1602. any heretical opinion contrary to the holy scripture of god and first four general counsel wherein the same be declare

# Step 2: Segmenting Texts into Chunks 

In [38]:
from pathlib import Path
import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

TEXT_DIR = Path("texts")
txt_paths = sorted(TEXT_DIR.glob("*.txt"))

sample_path = txt_paths[0]
# sample_path = "texts/A00419.txt"

with open(sample_path, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

# Split text into sentences
sentences = nltk.sent_tokenize(text)

print("File:", sample_path)
print("Number of sentences:", len(sentences))
print()

# Group sentences into chunks of ~120 words
TARGET_WORDS = 120

chunks = []
current = []
current_len = 0

for sent in sentences:
    words = sent.split()
    if not words:
        continue

    # If adding this sentence would exceed the target, finalize the chunk
    if current_len + len(words) > TARGET_WORDS and current:
        chunks.append(" ".join(current))
        current = []
        current_len = 0

    current.append(sent)
    current_len += len(words)

# Add any leftover sentences
if current:
    chunks.append(" ".join(current))

print("Number of chunks:", len(chunks))

# Diagnostics on chunk length (rough word counts)
lengths = [len(c.split()) for c in chunks]
lengths_sorted = sorted(lengths)

print()
print("Approx word counts per chunk:")
print("  min:", min(lengths))
print("  median:", lengths_sorted[len(lengths_sorted)//2])
print("  max:", max(lengths))

lo, hi = 5, 200
in_range = sum(lo <= n <= hi for n in lengths)
print(f"Chunks with {lo}–{hi} words:", in_range)
print("Share in range:", round(in_range / len(lengths), 3))

print()
print("--- Chunk 1 preview (first 400 chars) ---")
print(chunks[0][:400])

File: texts\A00169.txt
Number of sentences: 2

Number of chunks: 2

Approx word counts per chunk:
  min: 1094
  median: 3210
  max: 3210
Chunks with 5–200 words: 0
Share in range: 0.0

--- Chunk 1 preview (first 400 chars) ---
article to be inquire of in the diocese of chester in the ordinary visitation hold 1605. titul i. article concern religion prayer and sacrament first whether there be any abide in heretical opinion or resort to your parish that be know to have defend or maintain since the twenty day of march 1602. any heretical opinion contrary to the holy scripture of god and first four general counsel wherein th


# Step 3: Tokenize Chunks for Modelling

In [39]:
from pathlib import Path
import nltk
from gensim.utils import simple_preprocess

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

TEXT_DIR = Path("texts")
txt_paths = sorted(TEXT_DIR.glob("*.txt"))
# sample_path = txt_paths[0]
sample_path = "texts/A00419.txt"

with open(sample_path, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

# Step 2 logic: sentences -> chunks (~120 words)
sentences = nltk.sent_tokenize(text)

TARGET_WORDS = 120
chunks = []
current = []
current_len = 0

for sent in sentences:
    words = sent.split()
    if not words:
        continue

    if current_len + len(words) > TARGET_WORDS and current:
        chunks.append(" ".join(current))
        current = []
        current_len = 0

    current.append(sent)
    current_len += len(words)

if current:
    chunks.append(" ".join(current))

# Step 3: now we tokenize each chunk
token_lists = [simple_preprocess(c, deacc=True) for c in chunks]

print("File:", sample_path)
print("Chunks (strings):", len(chunks))
print("Chunks (token lists):", len(token_lists))

print("\n--- Token preview (first 60 tokens of first chunk) ---")
print(token_lists[0][:60])

print("\n--- Token count of first chunk ---")
print(len(token_lists[0]))

File: texts/A00419.txt
Chunks (strings): 5172
Chunks (token lists): 5172

--- Token preview (first 60 tokens of first chunk) ---
['the', 'first', 'booke', 'of', 'the', 'covntrie', 'farme', 'chap', 'what', 'manner', 'of', 'husbandrie', 'is', 'entreated', 'of', 'in', 'the', 'discourse', 'following', 'even', 'as', 'the', 'manner', 'of', 'building', 'vsed', 'at', 'this', 'day', 'the', 'varietie', 'of', 'countries', 'causeth', 'diuers', 'manner', 'of', 'labouring', 'of', 'the', 'earth']

--- Token count of first chunk ---
41


# Step 4: Process all Files and Filter Chunks

In [40]:
from pathlib import Path
import nltk
from gensim.utils import simple_preprocess

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

TEXT_DIR = Path("texts")
txt_paths = sorted(TEXT_DIR.glob("*.txt"))

TARGET_WORDS = 120
MIN_WORDS = 5
MAX_WORDS = 200

all_chunks = []
all_token_lists = []

def chunk_text(text, target_words=120):
    sentences = nltk.sent_tokenize(text)

    chunks = []
    current = []
    current_len = 0

    for sent in sentences:
        words = sent.split()
        if not words:
            continue

        if current_len + len(words) > target_words and current:
            chunks.append(" ".join(current))
            current = []
            current_len = 0

        current.append(sent)
        current_len += len(words)

    if current:
        chunks.append(" ".join(current))

    return chunks

print(f"Found {len(txt_paths)} text files.")
print()

for path in txt_paths:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    chunks = chunk_text(text, TARGET_WORDS)

    for c in chunks:
        tokens = simple_preprocess(c, deacc=True)
        n_tokens = len(tokens)

        # Filter by length
        if MIN_WORDS <= n_tokens <= MAX_WORDS:
            all_chunks.append(c)
            all_token_lists.append(tokens)

print("Total chunks kept (after filtering):", len(all_chunks))

Found 571 text files.

Total chunks kept (after filtering): 534933


# Step 5: Train Word2 Vec on our Corpus

In [41]:
import os
os.cpu_count()

20

In [42]:
from pathlib import Path
import nltk
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)


# Load and preprocess all texts (same logic as Step 4)


TEXT_DIR = Path("texts")
txt_paths = sorted(TEXT_DIR.glob("*.txt"))

TARGET_WORDS = 120
MIN_WORDS = 5
MAX_WORDS = 200

def chunk_text(text, target_words=120):
    sentences = nltk.sent_tokenize(text)

    chunks = []
    current = []
    current_len = 0

    for sent in sentences:
        words = sent.split()
        if not words:
            continue

        if current_len + len(words) > target_words and current:
            chunks.append(" ".join(current))
            current = []
            current_len = 0

        current.append(sent)
        current_len += len(words)

    if current:
        chunks.append(" ".join(current))

    return chunks

token_lists = []

print(f"Found {len(txt_paths)} text files.")
print("Building token lists...")

for path in txt_paths:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    chunks = chunk_text(text, TARGET_WORDS)

    for c in chunks:
        tokens = simple_preprocess(c, deacc=True)
        if MIN_WORDS <= len(tokens) <= MAX_WORDS:
            token_lists.append(tokens)

print("\nTotal tokenized chunks kept:", len(token_lists))

################
# Train Word2Vec (same logic as what we did in week 07)
################

print("\nTraining Word2Vec...")

model = Word2Vec(
    sentences=token_lists,
    vector_size=200,   # dimensionality of word vectors
    window=5,          # context window size
    min_count=5,       # ignore very rare words
    workers=20,         # adjust depending on your machine (see Week 07)
    sg=1               # 1 = skip-gram; 0 = CBOW
)

# Save model

Path("models").mkdir(exist_ok=True)
model_path = Path("models") / "w2v_full.bin"
model.save(str(model_path))

print("\nModel saved to:", model_path)

Found 571 text files.
Building token lists...

Total tokenized chunks kept: 534933

Training Word2Vec...

Model saved to: models\w2v_full.bin


## Step 5b

In [43]:
from pathlib import Path
from gensim.models import Word2Vec

model_path = Path("models") / "w2v_full.bin"
model = Word2Vec.load(str(model_path))

seed = "merchant"

if seed not in model.wv:
    print(f"'{seed}' not found in the model vocabulary.")
    print("This usually means min_count is too high or the corpus is too small.")
else:
    print(f"Top 30 words similar to '{seed}':")
    for word, score in model.wv.similar_by_word(seed, topn=30):
        print(f"  {word:20s} {score:.3f}")

Top 30 words similar to 'merchant':
  marchant             0.742
  factor               0.700
  merchants            0.685
  customer             0.659
  staplers             0.648
  clothier             0.644
  jeweller             0.644
  clothyer             0.642
  purser               0.641
  apprentice           0.638
  sailor               0.630
  tailor               0.628
  adventurer           0.624
  brewer               0.624
  seller               0.621
  haberdasher          0.621
  goldsmith            0.621
  tradesman            0.616
  venturer             0.616
  lete                 0.615
  marchants            0.613
  manufacturer         0.611
  wholesale            0.610
  vintner              0.607
  adventurers          0.607
  mariner              0.604
  stationer            0.600
  farmer               0.599
  gentleman            0.599
  withington           0.595


# Step 6: Build a tiered keyword list (A/B/C)

In [44]:
# Tier A: direct spellings / variants of the seed concept
TIER_A = {
    "merchant", "merchants",
    "marchant", "marchants"
}

# Tier B: closely related commercial roles / terms
TIER_B = {
    "factor", "chapman",
    "adventurer", "adventurers",
    "venturer", "venturers",
    "staple", "staplers",
    "trade",
    "purser"
}

# Tier C: "maybe" occupational neighborhood (often adjacent, not always merchant-specific)
TIER_C = {
    "clothier", "clothyer",
    "tailor", "tayler",
    "haberdasher",
    "goldsmith",
    "vintner",
    "brewer",
    "banker",
    "grazier",
    "jeweller"
}

print("Tier A size:", len(TIER_A))
print("Tier B size:", len(TIER_B))
print("Tier C size:", len(TIER_C))

print("\nTier A:", sorted(TIER_A))
print("\nTier B:", sorted(TIER_B))
print("\nTier C:", sorted(TIER_C))

Tier A size: 4
Tier B size: 10
Tier C size: 11

Tier A: ['marchant', 'marchants', 'merchant', 'merchants']

Tier B: ['adventurer', 'adventurers', 'chapman', 'factor', 'purser', 'staple', 'staplers', 'trade', 'venturer', 'venturers']

Tier C: ['banker', 'brewer', 'clothier', 'clothyer', 'goldsmith', 'grazier', 'haberdasher', 'jeweller', 'tailor', 'tayler', 'vintner']


# Step 7: Label chunks using tiered keywords (weak supervision)


In [45]:
from pathlib import Path
import nltk
from gensim.utils import simple_preprocess
import json

# nltk.download("punkt", quiet=True)
# nltk.download("punkt_tab", quiet=True)

#
# Tier definitions (from Step 6)


TIER_A = {
    "merchant", "merchants",
    "marchant", "marchants"
}

TIER_B = {
    "factor", "chapman",
    "adventurer", "adventurers",
    "venturer", "venturers",
    "staple", "staplers",
    "trade", "purser"
}

TIER_C = {
    "clothier", "clothyer",
    "tailor", "tayler",
    "haberdasher",
    "goldsmith",
    "vintner",
    "brewer",
    "banker",
    "grazier",
    "jeweller"
}


# Load and process texts


TEXT_DIR = Path("texts")
txt_paths = sorted(TEXT_DIR.glob("*.txt"))

TARGET_WORDS = 120
MIN_WORDS = 5
MAX_WORDS = 200

def chunk_text(text, target_words=120):
    sentences = nltk.sent_tokenize(text)

    chunks = []
    current = []
    current_len = 0

    for sent in sentences:
        words = sent.split()
        if not words:
            continue

        if current_len + len(words) > target_words and current:
            chunks.append(" ".join(current))
            current = []
            current_len = 0

        current.append(sent)
        current_len += len(words)

    if current:
        chunks.append(" ".join(current))

    return chunks

labeled = []

print(f"Processing {len(txt_paths)} files...")

for path in txt_paths:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    chunks = chunk_text(text, TARGET_WORDS)

    for c in chunks:
        tokens = simple_preprocess(c, deacc=True)
        n_tokens = len(tokens)

        if not (MIN_WORDS <= n_tokens <= MAX_WORDS):
            continue

        token_set = set(tokens)

        if token_set & (TIER_A | TIER_B):
            label = 1   # CORE
        elif token_set & TIER_C:
            label = 2   # MAYBE
        else:
            label = 0   # NEG

        labeled.append((c, label))

print("Total chunks labeled:", len(labeled))


# Save labeled data


Path("data").mkdir(exist_ok=True)

with open(Path("data") / "merchant_labeled_chunks.json", "w", encoding="utf-8") as f:
    json.dump(labeled, f, ensure_ascii=False)

print("Saved labeled chunks to data/merchant_labeled_chunks.json")

Processing 571 files...
Total chunks labeled: 534933
Saved labeled chunks to data/merchant_labeled_chunks.json


# Step 8: Sample and Inspect

In [46]:
print("\nLabel distribution:")
print(f"  CORE (1): {sum(1 for _, y in labeled if y == 1)}")
print(f"  NEG  (0): {sum(1 for _, y in labeled if y == 0)}")
print(f"  MAYBE(2): {sum(1 for _, y in labeled if y == 2)}")

# Sample one from each category
for label_name, label_val in [("CORE", 1), ("MAYBE", 2), ("NEG", 0)]:
    example = next((text for text, y in labeled if y == label_val), None)
    if example:
        print(f"\n{label_name} example (first 200 chars):")
        print(example[:200])


Label distribution:
  CORE (1): 9774
  NEG  (0): 524690
  MAYBE(2): 469

CORE example (first 200 chars):
toward the trade-way, you shall make a partition of tenne
or twelue furlongs, well inclosed with Ditch and Quickset, hedged round about, for
the feeding of your tyred, wearie, or sicke Cattell, which 

MAYBE example (first 200 chars):
I (...) therefore (...)
a baker, panter, worker in pastrie, and a brewer when need shall be (...),

that he should not be ignorant of any thing which might helpe to keepe, sustaine,
and inrich his hou

NEG example (first 200 chars):
the court of king JAMES after that I have resolve and with myself determine illustrious and thrice noble to a preamble wherein be brief discourse the cause of this treatise divulge and set forth unto 


# Step 9: Decide what to train on


In [47]:
import json
from pathlib import Path
import random

random.seed(42) # For reproducibility--same random sample each time [see (*) below]

DATA_PATH = Path("data") / "merchant_labeled_chunks.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    labeled = json.load(f)

core = [(t, 1) for (t, y) in labeled if y == 1]
neg  = [(t, 0) for (t, y) in labeled if y == 0]
maybe = [t for (t, y) in labeled if y == 2]

print("Loaded:")
print("  CORE:", len(core))
print("  NEG :", len(neg))
print("  MAYBE:", len(maybe))

###### => NEG step [see explanation below]
neg_sample = random.sample(neg, len(core))

training_data = core + neg_sample
random.shuffle(training_data)

print("Training set size (CORE + NEG):", len(training_data))

### Split the data into train and test sets:

split = int(0.8 * len(training_data))
train_data = training_data[:split]
test_data  = training_data[split:]

print("Train size:", len(train_data))
print("Test size :", len(test_data))
print("MAYBE size:", len(maybe))

# Save the datasets:

Path("data").mkdir(exist_ok=True)

with open(Path("data") / "train_core_vs_neg.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False)

with open(Path("data") / "test_core_vs_neg.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False)

with open(Path("data") / "maybe_texts.json", "w", encoding="utf-8") as f:
    json.dump(maybe, f, ensure_ascii=False)

print("Saved training, test, and MAYBE datasets.")

Loaded:
  CORE: 9774
  NEG : 524690
  MAYBE: 469
Training set size (CORE + NEG): 19548
Train size: 15638
Test size : 3910
MAYBE size: 469
Saved training, test, and MAYBE datasets.


# Step 10: TF-IDF

In [48]:
import json
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
DATA_DIR = Path("data")

#We now load the datasets we prepared at the end of last week (step 9)
with open(DATA_DIR / "train_core_vs_neg.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(DATA_DIR / "test_core_vs_neg.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

# Separate texts and labels
X_train_texts = [t for (t, y) in train_data]
y_train = [y for (t, y) in train_data]

X_test_texts = [t for (t, y) in test_data]
y_test = [y for (t, y) in test_data]

print("Train size:", len(X_train_texts))
print("Test size :", len(X_test_texts))

##### ==> See explanation [A] below

vectorizer = TfidfVectorizer(
    lowercase=True,
    min_df=5,        # ignore very rare words
    max_df=0.9       # ignore extremely common words; Explanation [B]
)
X_train = vectorizer.fit_transform(X_train_texts)
X_test = vectorizer.transform(X_test_texts)

print("TF-IDF matrix shapes:")
print("  Train:", X_train.shape)
print("  Test :", X_test.shape)

#####  ==> See explanation [C] below

Train size: 15638
Test size : 3910
TF-IDF matrix shapes:
  Train: (15638, 16198)
  Test : (3910, 16198)


# Step 11: Train Classifier


In [49]:
from pathlib import Path
import json

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from sklearn.feature_extraction.text import TfidfVectorizer

DATA_DIR = Path("data")

with open(DATA_DIR / "train_core_vs_neg.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(DATA_DIR / "test_core_vs_neg.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

X_train_texts = [t for (t, y) in train_data]
y_train = [y for (t, y) in train_data]

X_test_texts = [t for (t, y) in test_data]
y_test = [y for (t, y) in test_data]

vectorizer = TfidfVectorizer(
    lowercase=True,
    min_df=5,
    max_df=0.9
)

X_train = vectorizer.fit_transform(X_train_texts)
X_test = vectorizer.transform(X_test_texts)

clf = LogisticRegression(
    max_iter=1000,
)

clf.fit(X_train, y_train)

#test set predictions
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(cm)

print("\nClassification report:")
print(classification_report(y_test, y_pred))

###ROC AUC
auc = roc_auc_score(y_test, y_prob)
print("ROC AUC:", round(auc, 3))

print("\nNumber of non-zero coefficients:", (clf.coef_[0] != 0).sum())

top_pos_words = sorted(
    zip(vectorizer.get_feature_names_out(), clf.coef_[0]),
    key=lambda x: x[1],
    reverse=True
)[:15]

print("\nTop 15 positive words:")
for word, coef in top_pos_words:
    print(f"  {word}: {coef:.3f}")

top_neg_words = sorted(
    zip(vectorizer.get_feature_names_out(), clf.coef_[0]),
    key=lambda x: x[1]
)[:15]

print("\nTop 15 negative words:")
for word, coef in top_neg_words:
    print(f"  {word}: {coef:.3f}")

from pathlib import Path
import joblib

MODEL_DIR = Path.cwd() / "models"
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(vectorizer, MODEL_DIR / "tfidf_vectorizer.joblib")
joblib.dump(clf, MODEL_DIR / "merchant_logreg.joblib")

print("Saved TF-IDF vectorizer and classifier to /models/")

Confusion matrix:
[[1906   75]
 [ 131 1798]]

Classification report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95      1981
           1       0.96      0.93      0.95      1929

    accuracy                           0.95      3910
   macro avg       0.95      0.95      0.95      3910
weighted avg       0.95      0.95      0.95      3910

ROC AUC: 0.987

Number of non-zero coefficients: 16198

Top 15 positive words:
  trade: 27.556
  merchants: 22.286
  merchant: 17.624
  marchants: 13.415
  marchant: 8.709
  factor: 5.189
  staple: 4.985
  commodities: 4.328
  ship: 3.149
  buy: 2.995
  chapman: 2.957
  exchange: 2.949
  goods: 2.933
  ships: 2.726
  money: 2.701

Top 15 negative words:
  god: -3.122
  power: -1.928
  pope: -1.897
  rome: -1.888
  christ: -1.767
  spirit: -1.650
  enemies: -1.645
  gods: -1.627
  forces: -1.600
  faith: -1.593
  against: -1.571
  vertue: -1.506
  that: -1.434
  de: -1.353
  armie: -1.349
Saved TF-

# L1

In [50]:
from pathlib import Path
import json

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from sklearn.feature_extraction.text import TfidfVectorizer

DATA_DIR = Path("data")

with open(DATA_DIR / "train_core_vs_neg.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(DATA_DIR / "test_core_vs_neg.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

X_train_texts = [t for (t, y) in train_data]
y_train = [y for (t, y) in train_data]

X_test_texts = [t for (t, y) in test_data]
y_test = [y for (t, y) in test_data]

vectorizer = TfidfVectorizer(
    lowercase=True,
    min_df=5,
    max_df=0.9
)

X_train = vectorizer.fit_transform(X_train_texts)
X_test = vectorizer.transform(X_test_texts)

# clf = LogisticRegression(
#     max_iter=1000,
#     penalty='l1',
#     solver='liblinear'
# )
clf = LogisticRegression(
    max_iter=1000,
    l1_ratio=1,
    solver='liblinear'
)

clf.fit(X_train, y_train)

#test set predictions
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(cm)

print("\nClassification report:")
print(classification_report(y_test, y_pred))

###ROC AUC
auc = roc_auc_score(y_test, y_prob)
print("ROC AUC:", round(auc, 3))

print("\nNumber of non-zero coefficients:", (clf.coef_[0] != 0).sum())

top_pos_words = sorted(
    zip(vectorizer.get_feature_names_out(), clf.coef_[0]),
    key=lambda x: x[1],
    reverse=True
)[:15]

print("\nTop 15 positive words:")
for word, coef in top_pos_words:
    print(f"  {word}: {coef:.3f}")

top_neg_words = sorted(
    zip(vectorizer.get_feature_names_out(), clf.coef_[0]),
    key=lambda x: x[1]
)[:15]

print("\nTop 15 negative words:")
for word, coef in top_neg_words:
    print(f"  {word}: {coef:.3f}")

from pathlib import Path
import joblib

MODEL_DIR = Path.cwd() / "models"
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(vectorizer, MODEL_DIR / "tfidf_vectorizer_l1.joblib")
joblib.dump(clf, MODEL_DIR / "merchant_logreg_l1.joblib")

print("Saved TF-IDF vectorizer and classifier to /models/")

Confusion matrix:
[[1981    0]
 [   6 1923]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1981
           1       1.00      1.00      1.00      1929

    accuracy                           1.00      3910
   macro avg       1.00      1.00      1.00      3910
weighted avg       1.00      1.00      1.00      3910

ROC AUC: 1.0

Number of non-zero coefficients: 28

Top 15 positive words:
  trade: 174.931
  merchants: 148.667
  merchant: 123.939
  marchants: 90.009
  marchant: 66.278
  staple: 57.439
  factor: 56.940
  adventurers: 42.912
  chapman: 36.522
  adventurer: 31.384
  purser: 28.667
  venturers: 12.074
  ship: 3.073
  have: 1.823
  william: 1.700

Top 15 negative words:
  god: -2.722
  that: -2.560
  it: -0.856
  christ: -0.341
  not: -0.281
  00: 0.000
  000: 0.000
  10: 0.000
  1000: 0.000
  10000: 0.000
  100000: 0.000
  101: 0.000
  102: 0.000
  103: 0.000
  104: 0.000
Saved TF-IDF vectorizer a